# 01 · Ingest EUR/PLN exchange rates from the NBP API → bronze

Downloads official average exchange rates (table A) published by **Narodowy Bank Polski**
and stores them in the lakehouse:

* raw JSON response → `Files/raw/nbp/` (audit trail)
* table `bronze_nbp_fx_rates`

The NBP API returns at most 93 days per request, so the date range is split into chunks.
Weekends and public holidays have no rates – they are filled forward in the silver layer.

**Requires:** default lakehouse `lh_fpa` attached to this notebook.

In [ ]:
# Parameters (can be overridden from a data pipeline)
START_DATE = "2024-12-01"   # one month before the first invoice, so the "previous business day" rate always exists
END_DATE = ""               # empty = today
CURRENCIES = ["EUR"]

In [ ]:
import json
import os
from datetime import date, datetime, timedelta

import pandas as pd
import requests
from pyspark.sql import functions as F

LOCAL_FILES = "/lakehouse/default/Files/raw"
NBP_URL = "https://api.nbp.pl/api/exchangerates/rates/a/{cur}/{start}/{end}/?format=json"
MAX_DAYS = 93


def fetch_nbp_rates(currency: str, start: date, end: date) -> tuple[list[dict], list[dict]]:
    """Return (rows, raw_responses) for one currency, calling the API in 93-day chunks."""
    rows, raw = [], []
    chunk_start = start
    while chunk_start <= end:
        chunk_end = min(chunk_start + timedelta(days=MAX_DAYS - 1), end)
        url = NBP_URL.format(cur=currency.lower(), start=chunk_start.isoformat(), end=chunk_end.isoformat())
        resp = requests.get(url, headers={"Accept": "application/json"}, timeout=30)
        if resp.status_code == 404:          # "Brak danych" - no rates published in this range
            print(f"  {chunk_start} → {chunk_end}: no data")
        else:
            resp.raise_for_status()
            payload = resp.json()
            raw.append(payload)
            for r in payload["rates"]:
                rows.append({
                    "currency": payload["code"],
                    "table_no": r["no"],
                    "effective_date": r["effectiveDate"],
                    "mid_rate": r["mid"],
                })
            print(f"  {chunk_start} → {chunk_end}: {len(payload['rates'])} rates")
        chunk_start = chunk_end + timedelta(days=1)
    return rows, raw

In [ ]:
start = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end = datetime.strptime(END_DATE, "%Y-%m-%d").date() if END_DATE else date.today()

all_rows = []
for cur in CURRENCIES:
    print(f"Fetching {cur} {start} → {end}")
    rows, raw = fetch_nbp_rates(cur, start, end)
    all_rows.extend(rows)

    # keep the raw API response in Files for auditability
    os.makedirs(f"{LOCAL_FILES}/nbp", exist_ok=True)
    with open(f"{LOCAL_FILES}/nbp/nbp_{cur.lower()}_{start}_{end}.json", "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False)

pdf = pd.DataFrame(all_rows)
print(f"Downloaded {len(pdf)} rates")
pdf.tail()

In [ ]:
bronze = (
    spark.createDataFrame(pdf)
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn("mid_rate", F.col("mid_rate").cast("decimal(10,4)"))
    .withColumn("_loaded_at", F.current_timestamp())
)

(bronze.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_nbp_fx_rates"))

display(bronze.orderBy(F.desc("effective_date")).limit(10))